### Input Libraries

In [21]:
import json
import re
from langchain.schema import BaseOutputParser
from langchain.chains.router.llm_router import LLMRouterChain
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_community.llms import Ollama
import math

### Define LLM

In [101]:
# Specify the remote server's URL
llm = Ollama(model="deepseek-r1:1.5b-qwen-distill-q4_K_M", base_url="http://127.0.0.1:11434")


#### Using LLM chains for extracting parameters 

In [102]:
import re
import json
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_community.llms import Ollama


extraction_prompt = PromptTemplate(
    input_variables=["user_input"],
    template=(
        "User query:\n"
        "\"{user_input}\"\n\n"
        "Extract these fields as *pure JSON* (no markdown fences, no extra text):\n"
        "- deposit_amount (integer months: deposit)\n"
        "- loan_amount (integer montoan)\n"
        "- deposit_duration (integer months)\n"
        "- repayment_duration (integer months)\n"
        "- Credit_score (string or null)\n"
        "- Interest_rate (integer percent without % sign)\n\n"
        "If missing, set value to null.\n"
        "Numbers must be plain digits (e.g. 25000000), no underscores or commas.\n\n"
        "Example output:\n"
        "{{\n"
        '  "deposit_amount": 500000000,\n'
        '  "deposit_duration": 3,\n'
        '  "loan_amount": 150000000,\n'
        '  "repayment_duration": 36,\n'
        '  "Credit_score": "B",\n'
        '  "Interest_rate": 23\n'
        "}}\n"
    )
)


extraction_chain = LLMChain(
    llm=llm,
    prompt=extraction_prompt,
    verbose=False,
)

# 3) Helper to clean & load JSON
def clean_and_parse(raw: str) -> dict:
    # strip markdown fences if any
    cleaned = re.sub(r"```(?:json)?\s*", "", raw)
    cleaned = re.sub(r"```$", "", cleaned, flags=re.MULTILINE)
    # isolate the braces & their contents
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in LLM output:\n{repr(raw)}")
    js = match.group(0)
    # remove illegal underscores in numbers
    js = re.sub(r"(?<=\d)_(?=\d)", "", js)
    # remove trailing commas before closing brace/bracket
    js = re.sub(r",\s*([\}\]])", r"\1", js)
    # strip out any percent signs
    js = js.replace("%", "")
    # finally parse
    return json.loads(js)

# 4) Run end-to-end
user_input = "من میخوام ببینم سود سپرده بانک اگه 60700000 پول بخوابونم چقدره"
raw_output = extraction_chain.predict(user_input=user_input)
print("Raw LLM output:\n", raw_output)

try:
    params = clean_and_parse(raw_output)
    print("\nExtracted parameters:", params)
except Exception as e:
    print("❌ Parsing failed:", e)


Raw LLM output:
 <think>
Alright, let's tackle this query. The user wants me to extract some specific fields from a bank transaction and convert them into a JSON object. 

First, I'll look at the query again: it says "60700000" dollars deposited. So that's deposit_amount. I need to make sure it's an integer without any commas or underscores.

Next, they mention loan_amount but don't specify how much. It looks like this field might be missing data, so I'll set it to null.

Then there are two durations: deposit_duration and repayment_duration. Both are in months. The values given are 3 months each, so those should both be 3 in the JSON.

The Credit_score is a string or null. Since the user didn't provide this value, I'll default to "B".

Finally, the Interest_rate needs to be an integer without a percent sign. In the example, it was 23%, but the input here is 60700000. Without more info on the interest rate, I can only set it to null unless there's additional context.

I also need to ens

In [103]:
from langchain.prompts import PromptTemplate

extraction_prompt = PromptTemplate(
    input_variables=["user_input"],
    template=(
        # 1) absolutely forbid chain‐of‐thought
        "Do not think. Do not output any reasoning—output **only** the JSON.\n\n"

        # 2) define the fields
        "Extract exactly these fields as JSON (no markdown, no fences):\n"
        "- deposit_amount (integer or null)\n"
        "- loan_amount (integer or null)\n"
        "- deposit_duration (integer months or null)\n"
        "- repayment_duration (integer months or null)\n"
        "- Credit_score (string or null)\n"
        "- Interest_rate (integer percent without % or null)\n\n"

        # 3) if a field is not mentioned, it must be null
        "If missing, set its value to `null`. Numbers must be plain digits (e.g. 25000000),\n"
        "no underscores, commas, or % signs.\n\n"

        # 4) few‐shot example for missing durations
        "### Example 1\n"
        "Input: \"من میخوام ببینم سود سپرده بانک اگه 60700000 پول بخوابونم چقدره\"\n"
        "Output:\n"
        '{{'
        '"deposit_amount":60700000,'
        '"deposit_duration":null,'
        '"loan_amount":null,'
        '"repayment_duration":null,'
        '"Credit_score":null,'
        '"Interest_rate":null'
        '}}\n\n'

        # 5) now YOUR input
        "### Now process this input:\n"
        "Input: \"{user_input}\"\n"
        "Output:"
    )
)


In [104]:
import re
import json

def clean_and_parse(raw: str) -> dict:
    # 1) Strip any markdown fences (``` or ```json)
    cleaned = re.sub(r"```(?:json)?\s*\n?", "", raw)
    cleaned = re.sub(r"\n?```", "", cleaned)

    # 2) Now isolate the first {...} block in the remaining text
    match = re.search(r"\{[\s\S]*?\}", cleaned)
    if not match:
        raise ValueError(f"No JSON object found in LLM output:\n{raw!r}")
    js = match.group(0)

    # 3) Cleanup common issues:
    #    - trailing commas before a closing brace/bracket
    #    - percent signs
    #    - underscores in numbers
    js = re.sub(r",\s*([\}\]])", r"\1", js)
    js = js.replace("%", "")
    js = re.sub(r"(?<=\d)_(?=\d)", "", js)

    # 4) Finally, parse
    return json.loads(js)


In [ ]:
# # llm = Ollama(
# #     model="deepseek-r1:1.5b-qwen-distill-q4_K_M",
# #     base_url="http://127.0.0.1:11434",
# #     timeout=5,
# # )
# llm = Ollama(model="phi4:latest", base_url="http://127.0.0.1:11434")

In [105]:
from langchain_community.llms import Ollama
from langchain.chains import LLMChain


extraction_chain = LLMChain(
    llm=llm,
    prompt=extraction_prompt,
    verbose=False,
)

# user_input = "من میخوام ببینم سود سپرده بانک اگه 60700000 پول بخوابونم چقدره"
# user_input = "با ۳۵ میلیون تومن چقدر وام ۴ درصد میتونم بگیرم"
user_input = "من یه وام ۲۰۰ میلیونیه ۲ ساله میخوام"
raw = extraction_chain.predict(user_input=user_input)
print("🔴 Raw LLM output:\n", raw)

try:
    params = clean_and_parse(raw)
    print("✅ Extracted parameters:", params)
except Exception as e:
    print("❌ Parsing failed:", e)


🔴 Raw LLM output:
 <think>
Alright, let's take a look at the task. I need to extract specific fields from a given input and represent them in JSON format. The input is a sentence, and each field should be extracted with its corresponding value.

First, let's list out all the required fields:

- deposit_amount (integer or null)
- loan_amount (integer or null)
- deposit_duration (integer months or null)
- repayment_duration (integer months or null)
- Credit_score (string or null)
- Interest_rate (integer percent without % or null)

If any of these fields are missing in the input, their value should be set to `null`.

Now, looking at the example provided:

Input: "من میخوام ببینم سود سپرده بانک اگه 60700000 پول بخوابونم چقدره"

Output:
{"deposit_amount":60700000,"deposit_duration":null,"loan_amount":null,"repayment_duration":null,"Credit_score":null,"Interest_rate":null}

From this, I can see that the input successfully extracts all six fields. The deposit amount is extracted as a number,